# 3.5 SimdVF 与 SimtVF 的实现对照

## 概述

从 1.4 开始，我们一直在用 SimdVF。本节对 Add 算子分别通过 SimdVF 与 SimtVF 实现来进行对照，方便大家深入理解两种实现方式，并解释我们采用 Simd 作为示例的原因。对于 Simd 的计算部分，我们采用寄存器操作实现。

## 学习目标

- 说明 `vld → vadd → vsts` 与 GM/UB 搬运的联系与区别。
- 比较两种模式在相同输入上的计算表达。
- 理解向量寄存器与线程组织的区别，并将两种写法与 AIV 的执行方式联系起来。

[本章导航](README.md) · [上一节](03.04_pipeline.ipynb) · [下一节](03.06_softmax.ipynb)

<p><small>运行前完成第 1.4 节的环境准备，并使用本章 README 指定的已验证运行入口启动新内核。每页独立执行，工作目录为本章根目录；内核本身不初始化 NPU。</small></p>

## 1. SimdVF：用寄存器操作实现 Add

两份输入 A、B 和输出 C 都是连续 fp32 Tensor，计算为 `C[i] = A[i] + B[i]`。先用 `N=64, tile=64, blocks=1` 观察一个完整例子：GM 中的 A、B 先拷入 UB，再通过寄存器完成相加，最后将 UB 中的结果写回 GM。

### 先分清两层搬运

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">操作</th>
<th style="text-align: left; vertical-align: top;">从哪里到哪里</th>
<th style="text-align: left; vertical-align: top;">本例作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.copy(A[...], a)</code>、<code>T.copy(B[...], b)</code></td>
<td style="text-align: left; vertical-align: top;">GM → UB</td>
<td style="text-align: left; vertical-align: top;">将整个输入 tile 搬到核内</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.simd.vld(a[offset])</code>、<code>vld(b[offset])</code></td>
<td style="text-align: left; vertical-align: top;">UB → 向量寄存器</td>
<td style="text-align: left; vertical-align: top;">取出一组参与计算的元素</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.simd.vadd(ra, rb, mask)</code></td>
<td style="text-align: left; vertical-align: top;">寄存器输入 → 寄存器结果</td>
<td style="text-align: left; vertical-align: top;">相同 lane 的元素分别相加</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.simd.vsts(c[offset], rc, mask)</code></td>
<td style="text-align: left; vertical-align: top;">向量寄存器 → UB</td>
<td style="text-align: left; vertical-align: top;">将寄存器结果存入输出 buffer</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.copy(c, C[...])</code></td>
<td style="text-align: left; vertical-align: top;">UB → GM</td>
<td style="text-align: left; vertical-align: top;">将完整结果 tile 写回输出 Tensor</td>
</tr>
</tbody>
</table>

`vld` 的输入是 UB 地址，`vsts` 的目标也是 UB 地址；它们不能替代 GM 与 UB 之间的 `T.copy`。`vadd` 只做同位置相加，不会把所有 lane 归约成一个数。把这几步连起来，就是“GM → UB → 寄存器计算 → UB → GM”。

### 完整实现

以下两个版本共用输入与校验检查；这部分在 Host 上先校验不满足整除条件的配置。

```python
import tilelang
import tilelang.ascend.language as T


def check_add_config(N, tile, blocks, mode):
    if mode not in ("simd", "simt"):
        raise ValueError("mode must be simd or simt")
    if min(N, tile, blocks) <= 0 or tile % 64 or N % (tile * blocks):
        raise ValueError("Require tile % 64 == 0 and N % (tile * blocks) == 0")
    if mode == "simd" and 3 * tile * 4 > 248 * 1024:
        raise ValueError("SimdVF three fp32 buffers exceed the 248 KiB UB limit")
```

SIMD 版本显式申请三份 UB tile，因此检查其容量；本节 SIMT 版本直接访问 GM，不使用这三份 UB 内存。便于示例，这里 tile 取 64 的倍数。

```python
def add_simd(N=64, tile=64, blocks=1):
    check_add_config(N, tile, blocks, "simd")

    @T.prim_func
    def main(
        A: T.Tensor((N,), "float32"),
        B: T.Tensor((N,), "float32"),
        C: T.Tensor((N,), "float32"),
    ):
        with T.Kernel(blocks) as bx:
            a = T.alloc_shared((tile,), "float32")
            b = T.alloc_shared((tile,), "float32")
            c = T.alloc_shared((tile,), "float32")

            for it in T.serial(N // (blocks * tile)):
                begin = bx * (N // blocks) + it * tile
                T.copy(A[begin : begin + tile], a)
                T.copy(B[begin : begin + tile], b)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for group in T.serial(tile // 64):
                        ra = T.simd.vld(a[group * 64])
                        rb = T.simd.vld(b[group * 64])
                        rc = T.simd.vadd(ra, rb, mask)
                        T.simd.vsts(c[group * 64], rc, mask)

                T.copy(c, C[begin : begin + tile])

    return tilelang.compile(main, out_idx=-1)
```

输入 a、b 与输出 c 各占一份 UB，计算中间结果先保留在寄存器 rc 中。

## 2. SimtVF：直接访问 GM 实现 Add

SimtVF 可以直接访问 GM。本版本保留相同的输入输出与核间连续区间分配，不再申请 a、b、c 三份 UB tile，也不再使用 T.copy。每个核启动 `T.SimtVF(threads=2048)`，直接按索引读取 A/B 并写入 C：

```python
def add_simt(N=64, tile=64, blocks=1):
    check_add_config(N, tile, blocks, "simt")

    @T.prim_func
    def main(
        A: T.Tensor((N,), "float32"),
        B: T.Tensor((N,), "float32"),
        C: T.Tensor((N,), "float32"),
    ):
        with T.Kernel(blocks) as bx:
            begin = bx * (N // blocks)

            with T.SimtVF(threads=2048):
                for i in T.Parallel(N // blocks):
                    C[begin + i] = A[begin + i] + B[begin + i]

    return tilelang.compile(main, out_idx=-1)
```

`begin=bx*(N//blocks)` 是该核负责的起点，`T.Parallel(N//blocks)` 覆盖该核的全部元素。编译器将这些元素映射到执行线程；2048 是本节采用的线程配置，不是输入长度。主例只有 64 个有效元素，扩展用例每核有 32768 个有效元素，输出范围都由并行索引确定。

SIMD 版本用 tile 控制 UB 搬运并以 64 个 fp32 lane 为一组执行寄存器指令；SIMT 版本在一次执行区内直接并行处理核负责的 GM 区间，不再逐 tile 启动执行区。两种实现使用相同 A/B、输出与核数，但数据路径和线程组织不同，因此后面的性能比较是两种完整实现的比较，不能把差异只归因于指令表达方式。

两个版本的完整代码保存在 [add_modes.py](src/add_modes.py)。

## 3. 从硬件原理理解两种执行方式

前面的两种写法，对应 AIV（向量核）内 SIMD 和 SIMT 两种工作模式。VF（Vector Function）表示交给相应执行单元完成的一段计算任务。切换 VF 类型改变了核内的计算组织方式，并不是增加了一批物理核。

硬件结构和 VF 工作机制见 [Ascend C：抽象硬件架构](https://gitcode.com/cann/asc-devkit/blob/master/docs/zh/guide/programming_guide/advanced_programming/advanced_ai_core_programming_model/simd_simt_hybrid_programming/abstract_hardware_architecture.md)。

**SIMD：一条向量指令处理一组数据。** 本节的 `vld` 将 UB 数据载入向量寄存器，`vadd` 对寄存器中对应位置的数据并行相加，`vsts` 再将结果写回 UB。连续的多步计算可以复用寄存器中的中间结果，减少反复读写 UB。这里需要区分两层搬运：寄存器与 UB 之间的搬运属于 VF 内的操作，GM 与 UB 之间的搬运由 MTE 完成。这也解释了前面为什么同时需要 `vld/vsts` 和 `T.copy`。

详见 [Ascend C：Reg矢量计算编程](https://gitcode.com/cann/asc-devkit/blob/master/docs/zh/guide/programming_guide/programming_model/ai_core_simd_programming/c_pointer_programming/reg_vector_computation.md)。

**SIMT：按线程组织同一段计算。** 一个 SIMT VF 对应一个线程块，每 32 个线程组成一个 Warp。Warp 中的线程共享指令流，但各自保有线程索引和寄存器状态，因此可以用各自的索引访问不同数据。本例的 `threads=2048` 表示线程配置，不表示 2048 个物理核，也不保证所有线程在同一时刻执行。若同一 Warp 内的线程进入不同分支，硬件需要分批执行不同路径，可能降低有效并行度。

详见 [Ascend C：线程架构](https://gitcode.com/cann/asc-devkit/blob/master/docs/zh/guide/programming_guide/programming_model/ai_core_simt_programming/thread_architecture.md)。

回到 Add：SIMD 把连续元素组织成寄存器中的一组数据，SIMT 把元素工作分配给线程；两者都利用核内并行能力。本节 SIMT 代码直接读写 GM，省去了显式 UB tile 的管理，但仍有实际的访存和寄存器开销。因此，规则连续计算、离散索引和分支较多的计算，应结合各自的数据访问特点选择实现方式，再用实测判断效果。

以上链接来自 CANN 社区 asc-devkit 仓库 master 分支的 Ascend C 开发指南，用于延伸理解硬件和编程模型；其中的 Ascend C 接口名称与本章 TileLang 仅写法不同，硬件原理是一样的。

## 4. 两种模式的差异、性能与选用场景

![SimdVF 与直接访问 GM 的 SimtVF](images/03.05_modes_gm.svg)

左侧是 SimdVF 的 GM → UB → 寄存器计算 → UB → GM；右侧是 SimtVF 通过线程索引直接读写 GM。两边都计算 C=A+B，图中只展开少量位置。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">对照项</th>
<th style="text-align: left; vertical-align: top;">本节 SimdVF 实现</th>
<th style="text-align: left; vertical-align: top;">本节 SimtVF 实现</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">计算表达</td>
<td style="text-align: left; vertical-align: top;">显式向量寄存器和向量指令</td>
<td style="text-align: left; vertical-align: top;">并行索引直接读写 GM</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">计算代码</td>
<td style="text-align: left; vertical-align: top;">vld → vadd → vsts</td>
<td style="text-align: left; vertical-align: top;">C[begin+i]=A[begin+i]+B[begin+i]</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">计算组织</td>
<td style="text-align: left; vertical-align: top;">每组 64 个 fp32 lane，核内按 tile 循环</td>
<td style="text-align: left; vertical-align: top;">threads=2048，在核负责的完整区间上并行</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">显式管理内容</td>
<td style="text-align: left; vertical-align: top;">UB tile、寄存器分组、载入/存回</td>
<td style="text-align: left; vertical-align: top;">核起点、元素索引与线程配置</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">GM/UB 路径</td>
<td style="text-align: left; vertical-align: top;">T.copy 搬入 UB，计算后写回 GM</td>
<td style="text-align: left; vertical-align: top;">直接访问 GM，没有显式 T.copy</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">显式 UB tile 用量</td>
<td style="text-align: left; vertical-align: top;">3×tile×4 字节，受当前环境可分配的UB容量（248 KiB）约束</td>
<td style="text-align: left; vertical-align: top;">未申请 a/b/c tile；执行上下文仍由后端管理</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">性能判断</td>
<td style="text-align: left; vertical-align: top;">与同输入的另一完整实现实测比较</td>
<td style="text-align: left; vertical-align: top;">同左，不仅比较线程数或代码行数</td>
</tr>
</tbody>
</table>



**正确性验证**：两个规模、两种模式、四类输入共 16 组完整输出全部通过，最大绝对误差均为 0。



In [1]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)

def run_checked(script):
    try:
        result = subprocess.run([sys.executable, script], check=True,
                                capture_output=True, text=True)
    except subprocess.CalledProcessError as error:
        print(error.stdout)
        print(error.stderr)
        raise
    Path(OUTPUT_DIR / "validation", Path(script).stem + "_notebook.log").write_text(
        result.stdout + result.stderr)
    print("\n".join(line for line in result.stdout.splitlines()
                    if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

run_checked("src/run_add_modes.py")

SimdVF N=64 tile=64 blocks=1 random: full-output PASS; max_abs_error=0
SimdVF N=64 tile=64 blocks=1 zero: full-output PASS; max_abs_error=0
SimdVF N=64 tile=64 blocks=1 constant: full-output PASS; max_abs_error=0
SimdVF N=64 tile=64 blocks=1 ramp: full-output PASS; max_abs_error=0
SimtVF N=64 tile=64 blocks=1 random: full-output PASS; max_abs_error=0
SimtVF N=64 tile=64 blocks=1 zero: full-output PASS; max_abs_error=0
SimtVF N=64 tile=64 blocks=1 constant: full-output PASS; max_abs_error=0
SimtVF N=64 tile=64 blocks=1 ramp: full-output PASS; max_abs_error=0
SimdVF N=1179648 tile=16384 blocks=36 random: full-output PASS; max_abs_error=0
SimdVF N=1179648 tile=16384 blocks=36 zero: full-output PASS; max_abs_error=0
SimdVF N=1179648 tile=16384 blocks=36 constant: full-output PASS; max_abs_error=0
SimdVF N=1179648 tile=16384 blocks=36 ramp: full-output PASS; max_abs_error=0
SimtVF N=1179648 tile=16384 blocks=36 random: full-output PASS; max_abs_error=0
SimtVF N=1179648 tile=16384 blocks=36 

### 用 msprof 对比 kernel 耗时

先分别检查四类输入的完整输出，再比较相同 A、B 上的 kernel 耗时，使用 msprof `op_summary` 的 `Task Duration(us)`。

扩展用例选择 36 个向量核任务，每核处理 32768 项。这是本例的对比配置，不表示设备只有 36 个向量核；两种实现使用相同核数。SIMD 的 a、b、c 三份 fp32 UB 内存需要 `3×tile×4` 字节；按当前环境可分配的UB容量（248 KiB）和 64 项对齐计算，容量上限为 21120 项。

本节仍采用完整分块，tile 还必须整除每核的 32768 项，因此最大可用 tile 为 **16384**，三份 UB 内存合计 **192 KiB**，每核由 32 轮减为 2 轮。SIMT 保持直接访问 GM、2048 线程。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">规模</th>
<th style="text-align: left; vertical-align: top;">实现</th>
<th style="text-align: left; vertical-align: top;">N / SIMD tile / blocks</th>
<th style="text-align: left; vertical-align: top;">三批中位数/μs</th>
<th style="text-align: left; vertical-align: top;">kernel 中位数/μs</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">主例</td>
<td style="text-align: left; vertical-align: top;">SimdVF</td>
<td style="text-align: left; vertical-align: top;">64 / 64 / 1</td>
<td style="text-align: left; vertical-align: top;">0.7200, 0.7200, 0.6995</td>
<td style="text-align: left; vertical-align: top;">0.7180</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">主例</td>
<td style="text-align: left; vertical-align: top;">SimtVF，GM直访，2048线程</td>
<td style="text-align: left; vertical-align: top;">64 / 64 / 1</td>
<td style="text-align: left; vertical-align: top;">0.8560, 0.8535, 0.8205</td>
<td style="text-align: left; vertical-align: top;">0.8540</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">扩展</td>
<td style="text-align: left; vertical-align: top;">SimdVF</td>
<td style="text-align: left; vertical-align: top;">1179648 / 16384 / 36</td>
<td style="text-align: left; vertical-align: top;">5.3740, 5.3680, 5.3695</td>
<td style="text-align: left; vertical-align: top;">5.3705</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">扩展</td>
<td style="text-align: left; vertical-align: top;">SimtVF，GM直访，2048线程</td>
<td style="text-align: left; vertical-align: top;">1179648 / 16384 / 36</td>
<td style="text-align: left; vertical-align: top;">7.4750, 7.5090, 7.4745</td>
<td style="text-align: left; vertical-align: top;">7.4890</td>
</tr>
</tbody>
</table>

表中 tile 是 SIMD 的 UB 分块参数；SIMT 直接并行处理每核完整区间。小用例的 SimtVF/SimdVF 耗时比为 1.19，扩展用例为 1.39，课程参考测量的两组都是 SimdVF 更快。扩展用例比较的是增大 tile 后的 SIMD 与 GM 直访的 SIMT，不把差异只归因于执行模式。

```bash
source src/env.sh
for workload in small large; do
  for mode in simd simt; do
    msprof --output="${COURSE_OUTPUT_DIR}/profiles/msprof_simt_gm/replay_${workload}_${mode}" \
      --task-time=on --ai-core=off --runtime-api=on \
      python src/profile_add_modes.py "$mode" "$workload"
  done
done
```



### 怎样选择

- **连续元素的向量计算，且需要明确组织寄存器载入、计算和存回**：优先评估 SimdVF，按数据分组组织指令。
- **SimtVF**：适用于离散数据访问、复杂控制逻辑和分支发散等场景。
- **两种表达都适用时**：比较相同工作量下的完整输出、UB 用量和 msprof kernel 耗时，再决定采用哪一种。

本章选择 SimdVF 作为主线，是因为它便于把连续 tile、UB 数据、寄存器计算与 3.4 的流水联系起来，也能承接前面已有的实现。这个教学选择不等于已经证明它在所有算子上都比 SimtVF 快。
> 表中性能为课程参考测量记录，原始采集附件不在课程目录中。可按本节给出的命令重新采集当前环境的结果。

## 小结

同一个 Add 可以用寄存器指令，也可以用并行索引表达；本节 SimdVF 经 UB 和寄存器计算，SimtVF 通过线程索引直接访问 GM。下一节继续使用 SimdVF，把分块、分核、归约和流水组合成完整的行 Softmax。

## 课后练习

1. 为什么 `vsts(c[0], rc, mask)` 后仍然需要 `T.copy(c, C)`？
2. 将寄存器 Add 改成 Mul，保留载入、存回和 GM/UB 搬运，只把 `vadd` 换为 `vmul`。
3. `N=64`、64 个 fp32 lane、`threads=2048` 分别描述什么？

更多寄存器操作可继续阅读 [Ascend C：Reg矢量计算编程](https://gitcode.com/cann/asc-devkit/blob/master/docs/zh/guide/programming_guide/programming_model/ai_core_simd_programming/c_pointer_programming/reg_vector_computation.md)。

<details>
<summary>展开练习解析</summary>

1. vsts 只把寄存器结果写入 UB 中的 c，T.copy 才把 c 写回 GM 输出 C。

2. `rc = T.simd.vmul(ra, rb, mask)`，其余载入、存回和搬运保持不变；完整答案见 03.05_mul.py，已完成NPU 正确性校验。

3. N 是 Tensor 的元素总数；64 lane 是本例 fp32 向量寄存器的一组容量；threads=2048 是 SimtVF 执行区的线程配置。它们描述不同层次，不能互相替代。

</details>

In [2]:
run_checked("answer/03.05_mul.py")

Mul random: full-output PASS
Mul zero: full-output PASS
Mul constant: full-output PASS
Mul ramp: full-output PASS


[本章导航](README.md) · [上一节](03.04_pipeline.ipynb) · [下一节](03.06_softmax.ipynb)